# 06 – Export hold-out test predictions
Score the 50,000 hold-out students with the final model and save a CSV that advisors could use.

In [ ]:
import sys
from pathlib import Path
# make the shared module in ../src importable (no personal paths)
sys.path.append(str(Path.cwd().parent / 'src'))

In [ ]:
import joblib
import pandas as pd
from common import *

model = joblib.load(MODEL_DIR / "student_performance_model.joblib")
test = pd.read_csv(ROOT / "data" / "processed" / "test.csv", low_memory=False)
feats, _, _ = split_columns(test, True)
classes = list(model.named_steps["model"].classes_)
proba = model.predict_proba(test[feats])
out = pd.DataFrame({"student_id": test["student_id"], "actual_status": test[TARGET],
                    "predicted_status": [classes[i] for i in proba.argmax(axis=1)]})
for i, c in enumerate(classes):
    out[f"prob_{c}"] = proba[:, i].round(4)
out["risk_score"] = out[[f"prob_{c}" for c in AT_RISK]].sum(axis=1).round(4)
out["risk_level"] = pd.cut(out["risk_score"], [-0.01, 0.35, 0.60, 1.01], labels=["Low", "Medium", "High"])
out.to_csv(RES_DIR / "test_predictions.csv", index=False)
print(out.risk_level.value_counts())
print(pd.crosstab(out.risk_level, out.actual_status.isin(AT_RISK), normalize="index").round(3))